# 12 — FastAPI Serving Demo

Start the text-to-SQL API server on Colab and test it.
This notebook demonstrates the serving endpoint from `src/serving/app.py`.

**Runtime**: Google Colab T4 GPU

## 0. Setup

In [ ]:
!pip install -q transformers peft bitsandbytes accelerate torch
!pip install -q fastapi uvicorn requests

In [ ]:
!git clone https://github.com/oLittle-five/text-to-sql-llama.git
%cd text-to-sql-llama

In [ ]:
from huggingface_hub import login
login()

## 1. Start the API Server

We run the server in a background thread so we can test it from the same notebook.

In [ ]:
import threading
import uvicorn
import time

# Import the app
from src.serving.app import app, load_model

# Load model first (so we can see the progress)
print("Loading model...")
load_model()
print("Model loaded!")

In [ ]:
# Start server in background thread
def run_server():
    uvicorn.run(app, host="0.0.0.0", port=8000, log_level="warning")

server_thread = threading.Thread(target=run_server, daemon=True)
server_thread.start()
time.sleep(2)  # wait for server to start
print("Server running at http://localhost:8000")

## 2. Test the API

In [ ]:
import requests
import json

BASE_URL = "http://localhost:8000"

# Health check
resp = requests.get(f"{BASE_URL}/health")
print("Health check:")
print(json.dumps(resp.json(), indent=2))

In [ ]:
# Single prediction
resp = requests.post(f"{BASE_URL}/predict", json={
    "question": "How many people live in Tokyo?",
    "columns": ["city", "country", "population"],
    "types": ["text", "text", "real"],
})
result = resp.json()
print(f"SQL: {result['sql']}")
print(f"Time: {result['generation_time_ms']:.0f}ms")
print(f"Tokens: {result['prompt_tokens']} prompt + {result['generated_tokens']} generated")

In [ ]:
# Batch prediction
resp = requests.post(f"{BASE_URL}/predict/batch", json={
    "queries": [
        {
            "question": "What is the nationality of Terrence Ross?",
            "columns": ["Player", "No.", "Nationality", "Position", "Years in Toronto", "School/Club Team"],
            "types": ["text", "text", "text", "text", "text", "text"],
        },
        {
            "question": "How many schools or teams had Jalen Rose?",
            "columns": ["Player", "No.", "Nationality", "Position", "Years in Toronto", "School/Club Team"],
            "types": ["text", "text", "text", "text", "text", "text"],
        },
        {
            "question": "What was the date of the race in Misano?",
            "columns": ["No", "Date", "Circuit", "Pole Position", "Race winner"],
            "types": ["real", "text", "text", "text", "text"],
        },
    ]
})
result = resp.json()
for i, r in enumerate(result["results"]):
    print(f"Q{i+1}: {r['sql']}  ({r['generation_time_ms']:.0f}ms)")
print(f"\nTotal: {result['total_time_ms']:.0f}ms")

In [ ]:
# Validation error test
resp = requests.post(f"{BASE_URL}/predict", json={
    "question": "test",
    "columns": ["col1", "col2"],
    "types": ["text"],  # mismatched length
})
print(f"Status: {resp.status_code}")
print(f"Error: {resp.json()['detail']}")

## 3. Example: Interactive Query

Try your own questions!

In [ ]:
# Change these to try your own queries!
question = "Which country has the highest GDP?"
columns = ["Country", "GDP (billions)", "Population", "Continent"]
types = ["text", "real", "real", "text"]

resp = requests.post(f"{BASE_URL}/predict", json={
    "question": question,
    "columns": columns,
    "types": types,
})
result = resp.json()
print(f"Question: {question}")
print(f"Columns:  {', '.join(columns)}")
print(f"SQL:      {result['sql']}")
print(f"Time:     {result['generation_time_ms']:.0f}ms")

## 4. API Documentation

The API auto-generates interactive documentation:
- **Swagger UI**: http://localhost:8000/docs
- **ReDoc**: http://localhost:8000/redoc

(These are accessible when running locally but not from Colab's browser. 
The `curl` examples below show the raw API usage.)

In [ ]:
# curl-equivalent examples for README
print("""Example curl commands (for when running locally or deployed):

# Health check
curl http://localhost:8000/health

# Single prediction
curl -X POST http://localhost:8000/predict \\
  -H "Content-Type: application/json" \\
  -d '{
    "question": "How many people live in Tokyo?",
    "columns": ["city", "country", "population"],
    "types": ["text", "text", "real"]
  }'

# Batch prediction
curl -X POST http://localhost:8000/predict/batch \\
  -H "Content-Type: application/json" \\
  -d '{"queries": [{"question": "...", "columns": [...], "types": [...]}]}'
""")